# BedSense AI — Elderly Bed-Monitoring System
### End-to-End Pipeline: Primary Vision + Temporal State Engine + Agentic Reasoning + Policy

This notebook implements the complete 8-stage pipeline from **`Bed Monitoring System — Design Spec.md`**:
1. **Stage 1**: Setup & Configuration (`configs/thresholds.yaml`)
2. **Stage 2**: Primary Vision Smoke Test (YOLO-Pose tracking + 17 COCO keypoints)
3. **Stage 3**: Dynamic Bed Relation & Overlap Calculation
4. **Stage 4**: Posture Geometry Classification & Temporal Smoothing (`StateSegment`s)
5. **Stage 5**: Bed-Exit / Return Pattern Matcher & Confidence Routing
6. **Stage 6**: Agent LLM (Qwen3-4B / Qwen2.5) & Tools via Ollama in Docker
7. **Stage 7**: Alert Policy Engine & Report Generation (`timeline.json`, `events.json`, `summary.json`)
8. **Stage 8**: Quantitative Evaluation & Metrics against Ground Truth
9. **Stage 9**: Failure Case Analysis & Discussion

## Stage 1: Setup & Configuration

In [1]:
import os
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Ensure project root is in python path
root_dir = Path.cwd()
if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.config import SystemConfig
from src.constants import ALL_STATES, ALL_DECISIONS
from src.contracts import FrameObservation, StateSegment, BedEvent
from src.video import VideoSource
from src.perception import PerceptionDetector, BedRelationEngine
from src.state import FrameClassifier, TransitionSmoother, BedPatternMatcher, ConfidenceEvaluator
from src.agent import OllamaClient, ToolEnvironment, AgentDecisionLoop
from src.policy import PolicyEngine, ReportGenerator, format_seconds_to_hms
from src.eval import EvaluationMetrics, GroundTruthHandler
from src.pipeline import BedSensePipeline

config = SystemConfig.from_yaml("configs/thresholds.yaml")
print("System Configuration Loaded successfully.")
print(f"Device: {config.perception.device} | Pose Model: {config.perception.pose_model}")
print(f"Ollama Endpoint: {config.agent.ollama_base_url} | Text Model: {config.agent.text_model}")

System Configuration Loaded successfully.
Device: cuda:0 | Pose Model: yolo11n-pose.pt
Ollama Endpoint: http://ollama:11434 | Text Model: qwen2.5:3b


## Stage 2: Primary Vision Smoke Test
Single YOLO-Pose model execution extracting person detections, ByteTrack persistent IDs, and 17 COCO keypoints.

In [3]:
video_path = "old/data/sample2.mp4"
if not Path(video_path).exists():
    print(f"Sample video not found at {video_path}")
else:
    source = VideoSource(video_path, target_fps=10.0)
    detector = PerceptionDetector(
        model_path=config.perception.pose_model,
        conf_threshold=config.perception.pose_conf,
        device=config.perception.device,
    )
    
    # Process first 5 frames
    for i, sample in enumerate(source):
        obs = detector.get_primary_observation(sample.frame, sample.timestamp)
        print(f"Frame {i} (t={obs.t:.2f}s) | Track ID: {obs.track_id} | BBox: {obs.bbox} | Mean Kpt Conf: {obs.mean_kpt_conf:.2f}")
        if i >= 4:
            break
    source.release()

Frame 0 (t=0.00s) | Track ID: 1 | BBox: (479.5989074707031, 130.9578857421875, 776.2373046875, 581.6591796875) | Mean Kpt Conf: 0.91
Frame 1 (t=0.14s) | Track ID: 1 | BBox: (479.65899658203125, 130.9945831298828, 776.3675537109375, 581.8081665039062) | Mean Kpt Conf: 0.91
Frame 2 (t=0.23s) | Track ID: 1 | BBox: (479.6562194824219, 131.20130920410156, 776.2244262695312, 581.788818359375) | Mean Kpt Conf: 0.91
Frame 3 (t=0.32s) | Track ID: 1 | BBox: (479.73126220703125, 131.30445861816406, 776.2196655273438, 581.7664184570312) | Mean Kpt Conf: 0.91
Frame 4 (t=0.41s) | Track ID: 1 | BBox: (479.6513366699219, 131.33209228515625, 776.1868896484375, 581.8218383789062) | Mean Kpt Conf: 0.91


## Stage 3: Dynamic Bed Relation & Overlap Calculation
Dynamically detects patient bed coordinates from scene calculations and computes continuous geometric overlap.

In [4]:
bed_engine = BedRelationEngine(model_path="yolo11n.pt", device=config.perception.device)
bed_poly = bed_engine.auto_calibrate_from_video(video_path, sample_frames=15)

print("Calibrated Bed Bounding Box:", bed_engine.bbox)
print("Bed Polygon Coordinates:\n", bed_poly)

# Test overlap on sample coordinates
sample_in_bed_bbox = (bed_engine.bbox[0] + 50, bed_engine.bbox[1] + 50, bed_engine.bbox[2] - 50, bed_engine.bbox[3] - 50)
sample_out_of_bed_bbox = (10.0, 10.0, 100.0, 200.0)

overlap_in = bed_engine.compute_overlap(sample_in_bed_bbox)
overlap_out = bed_engine.compute_overlap(sample_out_of_bed_bbox)

print(f"In-Bed Overlap Calculation:     {overlap_in*100:.1f}%")
print(f"Out-of-Bed Overlap Calculation: {overlap_out*100:.1f}%")

Calibrated Bed Bounding Box: (359.29571533203125, 94.71060180664062, 926.1220703125, 709.7116088867188)
Bed Polygon Coordinates:
 [[      359.3      94.711]
 [     926.12      94.711]
 [     926.12      709.71]
 [      359.3      709.71]]
In-Bed Overlap Calculation:     100.0%
Out-of-Bed Overlap Calculation: 0.0%


## Stage 4: Frame Classifier & Transition Smoothing
Maps posture geometry × bed overlap into the 7 states, then applies `MIN_SEGMENT_SEC` (2.0s) smoothing to eliminate noise.

In [5]:
classifier = FrameClassifier()
smoother = TransitionSmoother(min_segment_sec=2.0)

source = VideoSource(video_path, target_fps=10.0)
raw_records = []
for idx, sample in enumerate(source):
    obs = detector.get_primary_observation(sample.frame, sample.timestamp)
    state, conf = classifier.classify(obs)
    raw_records.append((obs.t, state, conf, obs))
    if idx >= 100:
        break
source.release()

smoothed_segments = smoother.smooth(raw_records)
print(f"Generated {len(smoothed_segments)} smoothed StateSegments:")
for s in smoothed_segments:
    print(f"  [{format_seconds_to_hms(s.start_t)} -> {format_seconds_to_hms(s.end_t)}] State: {s.state:<20} | Duration: {s.duration:.1f}s | Conf: {s.confidence:.2f}")

Generated 2 smoothed StateSegments:
  [00:00:00 -> 00:00:03] State: STANDING             | Duration: 2.7s | Conf: 0.87
  [00:00:03 -> 00:00:10] State: WALKING              | Duration: 7.4s | Conf: 0.80


## Stage 5: Pattern Matcher & Confidence Scoring (Router Split)
State machine checks for candidate bed-exit / bed-return sequences while rejecting benign posture adjustments.
Evaluates the 5 ambiguity triggers (§4.1).

In [6]:
pattern_matcher = BedPatternMatcher(min_exit_confirm_sec=3.0, return_window_sec=30.0)
confidence_evaluator = ConfidenceEvaluator()

candidates = pattern_matcher.find_candidates(smoothed_segments)
print(f"Detected {len(candidates)} candidate bed events:")
for c in candidates:
    print(f"  Candidate {c.event_type.upper()} at {c.confirmed_t:.1f}s | Transition: {c.prev_state} -> {c.curr_state} | Ambiguous: {c.is_ambiguous}")

print("\nEvaluating Ambiguity Triggers for Segments:")
for s in smoothed_segments:
    s_obs = [o for _, _, _, o in raw_records if s.start_t <= o.t <= s.end_t]
    is_amb, conf, reason = confidence_evaluator.evaluate_segment(s, s_obs, candidates)
    route = "--> ROUTE TO AGENT LLM" if is_amb else "--> DIRECT TO POLICY"
    print(f"  [{s.state}] Ambiguous: {is_amb:<5} {route} {f'({reason})' if reason else ''}")

Detected 0 candidate bed events:

Evaluating Ambiguity Triggers for Segments:
  [STANDING] Ambiguous: 0     --> DIRECT TO POLICY 
  [WALKING] Ambiguous: 1     --> ROUTE TO AGENT LLM (Multiple track IDs {0, 1, 2} detected in single segment (caregiver presence / ID switch))


## Stage 6: Agent LLM & Tools (Ollama via Docker)
Multi-turn agentic decision loop with strict tool cap (`TOOL_CALL_LIMIT = 4`) and audit trail (`tool_trace`).

In [7]:
ollama = OllamaClient(base_url=config.agent.ollama_base_url)
print(f"Ollama Available: {ollama.is_available()} at {config.agent.ollama_base_url}")

tool_env = ToolEnvironment(
    all_segments=smoothed_segments,
    all_observations=[o for _, _, _, o in raw_records],
    ollama_client=ollama,
    fps=10.0,
)

# Test tool execution
overlap_tool_res = tool_env.check_bed_overlap(t=2.0)
print(f"Tool 'check_bed_overlap(t=2.0)' output: {overlap_tool_res}")

agent_loop = AgentDecisionLoop(ollama_client=ollama, tool_call_limit=4)
if smoothed_segments:
    res = agent_loop.resolve_ambiguity(smoothed_segments[0], tool_env)
    print("\nAgent Resolution Result:")
    print(f"  State: {res.state} | Conf: {res.confidence:.2f} | Event: {res.event}")
    print(f"  Reasoning: {res.reasoning}")
    print(f"  Tool Trace Count: {len(res.tool_trace)}")

Ollama Available: True at http://ollama:11434
Tool 'check_bed_overlap(t=2.0)' output: 0.0

Agent Resolution Result:
  State: STANDING | Conf: 0.70 | Event: None
  Reasoning: The patient was observed to be in an 'OUT_OF_BED' state with a high confidence, indicating they are likely outside the bed area and standing.
  Tool Trace Count: 2


## Stage 7: End-to-End Pipeline & JSON Report Generation
Executes full pipeline and exports `timeline.json`, `events.json`, and `summary.json` matching §6.2 schemas exactly.

In [ ]:
pipeline = BedSensePipeline(config=config)
result = pipeline.process_video(
    video_source=video_path,
    max_frames=120,
    output_json_dir="outputs/notebook_run",
)

print("=== SUMMARY.JSON ===")
import json
print(json.dumps(result.summary_json, indent=2))

print("\n=== EVENTS.JSON ===")
print(json.dumps(result.events_json, indent=2))

# Verify duration invariant
sum_dur = sum(result.summary_json["activity_duration_sec"].values())
obs_dur = result.summary_json["observation_duration_sec"]
assert sum_dur == obs_dur, f"Invariant violated: sum {sum_dur} != obs {obs_dur}"
print(f"\n✓ ACCEPTANCE CRITERIA VERIFIED: sum(activity_duration_sec) ({sum_dur}s) == observation_duration_sec ({obs_dur}s)")

## Stage 8: Evaluation & Quantitative Metrics
Computes state accuracy, confusion matrix, event precision/recall, and duration error against ground truth annotations.

In [ ]:
# Create ground truth template and evaluate
gt_df = GroundTruthHandler.create_template_csv("outputs/sample_gt.csv", duration_sec=result.summary_json["observation_duration_sec"])
metrics = EvaluationMetrics.compute_state_metrics(result.segments, gt_df)

print(f"Overall State Classification Accuracy: {metrics['accuracy']*100:.1f}%")
print("\nPer-Class Metrics:")
for label, m in metrics["class_metrics"].items():
    print(f"  {label:<20}: Prec: {m['precision']:.2f}, Rec: {m['recall']:.2f}, F1: {m['f1_score']:.2f}")

## Stage 9: Failure Case Analysis & Discussion (§11)

1. **Failure Case 1: Heavy Blanket / Quilt Occlusion**
   - *Observation*: When patient is tucked under thick blankets, lower limb keypoints (knees, ankles) drop in confidence below 0.20.
   - *System Mitigation*: Primary vision computes `mean_kpt_conf < 0.40`, triggering the §4.1 ambiguity rule. The Agent LLM re-queries recent torso geometry and queries `vlm_describe` to confirm patient presence.

2. **Failure Case 2: Caregiver Assisting Bed Exit (Track Occlusion & ID Switch)**
   - *Observation*: A caregiver enters the frame and stands near the bed, causing temporary bounding box overlap and potential track ID swapping.
   - *System Mitigation*: Trigger 5 detects multi-person track IDs in a single segment. Policy engine suppresses alert escalation from ALERT to MONITOR due to caregiver presence.

3. **Failure Case 3: Sitting on Bed Edge Hovering**
   - *Observation*: Patient sits on the mattress edge with bed overlap hovering between 0.45 and 0.55 for an extended period.
   - *System Mitigation*: Trigger 3 flags boundary hovering and routes to Agent LLM. If sustained > 300s, Policy engine elevates decision to `MONITOR` for clinical fall prevention.